# Exercise 01 — Your First Parallel Neuron Kernel

**Module 01 | Estimated time: 45–90 minutes**

---

## Background

In neuroscience simulations, we often need to apply the **same operation to every neuron simultaneously**. A simple but important case is computing an input current for each neuron at each timestep:

$$I_i = I_{\text{bias}} + \sigma \cdot \xi_i$$

where $\xi_i$ is a random noise term drawn independently for each neuron $i$.

For simplicity, in this exercise you will implement the **deterministic** version (no noise yet):

$$I_i = I_{\text{bias}} + w_i \cdot x_i$$

This is a **weighted input**: each neuron $i$ receives a bias current plus a weight $w_i$ times an input signal $x_i$. In a real network, $w_i$ is the synaptic weight and $x_i$ is the pre-synaptic activity.

---

## Task Overview

You will write a CUDA kernel that computes:
```
I[i] = I_bias + W[i] * X[i]    for all i = 0, ..., N-1
```
and verify it against a CPU reference.

**Parts:**
1. Fill in the kernel function
2. Fill in the memory management code
3. Fill in the launch configuration
4. Verify correctness and measure bandwidth
5. *(Challenge)* Extend to compute V update using I

In [ ]:
!nvidia-smi

## Part 1: Write the Kernel

Fill in the `???` sections below. Do not change anything else.

In [ ]:
%%writefile compute_current.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                        \
    cudaError_t e = (call);                                          \
    if (e != cudaSuccess) {                                          \
        fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(e)); \
        exit(1); }                                                   \
} while(0)

// ─────────────────────────────────────────────────────────────────────────────
// TODO Part 1: Complete the kernel
//
// Each thread should:
//   1. Compute its global index i
//   2. Check bounds (return if i >= N)
//   3. Compute I[i] = I_bias + W[i] * X[i]
// ─────────────────────────────────────────────────────────────────────────────
__global__ void compute_input_current(
    const float* W,      // synaptic weights [N]
    const float* X,      // pre-synaptic activity [N]
    float* I,            // output current [N]
    float I_bias,        // constant bias (scalar)
    int N                // number of neurons
) {
    // Step 1: compute global thread index
    int i = ???;

    // Step 2: bounds check
    if (???) return;

    // Step 3: compute current for neuron i
    I[i] = ???;
}

// CPU reference (for correctness checking)
void compute_current_cpu(const float* W, const float* X, float* I,
                          float I_bias, int N) {
    for (int i = 0; i < N; i++) {
        I[i] = I_bias + W[i] * X[i];
    }
}

int main() {
    const int N = 50000;      // neurons
    const float I_bias = 0.5f;
    size_t bytes = N * sizeof(float);

    // Host arrays
    float* h_W = (float*)malloc(bytes);
    float* h_X = (float*)malloc(bytes);
    float* h_I = (float*)malloc(bytes);      // GPU result
    float* h_Iref = (float*)malloc(bytes);   // CPU reference

    // Initialize
    for (int i = 0; i < N; i++) {
        h_W[i] = 0.1f * (i % 10);    // weights 0.0 to 0.9
        h_X[i] = (float)(i % 100) / 100.0f;  // activity 0.0 to 0.99
    }

    // ─────────────────────────────────────────────────────────────────────────
    // TODO Part 2: Allocate device memory for d_W, d_X, d_I
    // and copy h_W and h_X to the device
    // ─────────────────────────────────────────────────────────────────────────
    float *d_W, *d_X, *d_I;

    // Your cudaMalloc calls here:
    ???

    // Your cudaMemcpy Host->Device calls here:
    ???

    // ─────────────────────────────────────────────────────────────────────────
    // TODO Part 3: Set launch configuration and launch the kernel
    // Use 256 threads per block
    // ─────────────────────────────────────────────────────────────────────────
    int threads_per_block = ???;
    int num_blocks = ???;     // ceiling division

    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0));
    CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    // Your kernel launch here:
    compute_input_current<<<???>>(d_W, d_X, d_I, I_bias, N);

    CUDA_CHECK(cudaEventRecord(t1));
    CUDA_CHECK(cudaEventSynchronize(t1));
    float ms;
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));

    // Copy result back
    CUDA_CHECK(cudaMemcpy(h_I, d_I, bytes, cudaMemcpyDeviceToHost));

    // CPU reference
    compute_current_cpu(h_W, h_X, h_Iref, I_bias, N);

    // ─────────────────────────────────────────────────────────────────────────
    // TODO Part 4: Check correctness
    // Compute max absolute error between h_I and h_Iref
    // Print PASS if max_err < 1e-5, FAIL otherwise
    // ─────────────────────────────────────────────────────────────────────────
    float max_err = 0.0f;
    for (int i = 0; i < N; i++) {
        float err = ???;
        if (err > max_err) max_err = err;
    }

    double bw = 3.0 * bytes / (ms * 1e-3) / 1e9;  // reads W, X + writes I
    printf("N=%d neurons\n", N);
    printf("Kernel time:  %.3f ms\n", ms);
    printf("Bandwidth:    %.1f GB/s\n", bw);
    printf("Max error:    %.2e  %s\n", max_err, max_err < 1e-5f ? "PASS" : "FAIL");

    // TODO Part 2b: Free device memory
    ???

    free(h_W); free(h_X); free(h_I); free(h_Iref);
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    return 0;
}

In [ ]:
# Compile and run — fix errors until you see PASS
!nvcc -O2 -o compute_current compute_current.cu -lm && ./compute_current

## Part 5 — Challenge: Add the Voltage Update

Extend the program to also update membrane voltages using a simple Euler step:

$$V_i(t + \Delta t) = V_i(t) + \frac{\Delta t}{\tau_m} \left( -(V_i(t) - E_L) + R_m \cdot I_i \right)$$

Parameters:
- $\tau_m = 20$ ms (membrane time constant)
- $E_L = -65$ mV (leak reversal potential)
- $R_m = 10$ MΩ (membrane resistance)
- $\Delta t = 0.1$ ms
- Initial $V_i = -65$ mV (all neurons at rest)

Write a **second kernel** `update_voltage` and chain it after `compute_input_current`.

**Hint:** The two kernels run sequentially in the same CUDA stream, so no extra synchronization is needed between them.

In [ ]:
# Your Part 5 code here
%%writefile voltage_update.cu
// TODO: add #includes, CUDA_CHECK, and both kernels
// compute_input_current + update_voltage
// Run 1000 timesteps and print V[0] at each 100th step


In [ ]:
!nvcc -O2 -o voltage_update voltage_update.cu -lm && ./voltage_update

## Reflection Questions

Answer these in the cell below:

1. What does the `if (i >= N) return;` guard prevent? What would happen without it?
2. You used 256 threads per block. What happens if you change it to 32? To 1024? To 1025?
3. The `compute_input_current` kernel reads two arrays (W, X) and writes one (I). How many bytes per neuron does it transfer? What is the theoretical peak bandwidth you could achieve?
4. In the challenge, why can you call two kernels back-to-back without `cudaDeviceSynchronize` in between?

*Your answers here:*

1. 
2. 
3. 
4. 

---

When you are done, check your work against [ex01_solution.ipynb](ex01_solution.ipynb).